# PlateVision AI Pro — CNN V3 | Google Colab

**Objectif :** entraîner de zéro un réseau convolutif de classification de **36 caractères isolés (0–9, A–Z)** sur des données synthétiques générées à la volée, avec séparation des **groupes de familles de polices** entre entraînement, validation et test.

**Démarrage :** dans Colab, sélectionner **Exécution → Modifier le type d’exécution → GPU T4** (si disponible), puis **Exécuter tout**. La première exécution entraîne V3, produit des métriques sur la **validation** et prépare une archive. **Le test final est désactivé par défaut** pour éviter de l’utiliser dans le choix de l’architecture.

**Conservation :** ce notebook est autonome : il **ne charge ni n’écrase V1/V2**, n’utilise pas EasyOCR et écrit les résultats dans un dossier V3 unique.

**Limite :** les chiffres issus de ce notebook sont des performances sur des **caractères synthétiques**, pas des performances de reconnaissance de plaques réelles. Un modèle de caractère isolé ne détecte pas la plaque ni ne segmente ses caractères.

## 1 — Configuration et dépendances

Colab fournit généralement TensorFlow, Pillow, scikit-learn et Matplotlib. La cellule suivante indique les versions et les options utiles.

- `RUN_TRAIN = True` : entraînement V3 lors de l'exécution complète.
- `RUN_FINAL_TEST = False` : garde située dans la cellule du test final ; à déverrouiller uniquement après validation.
- Le jeu de données contient **14 400 / 3 600 / 3 600** caractères (train / validation / test).

In [ ]:
from pathlib import Path
from collections import defaultdict, Counter
from datetime import datetime, timezone
from io import BytesIO
import hashlib, json, math, os, random, string, sys, uuid, zipfile, warnings

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw, ImageFont, ImageFilter, ImageEnhance
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
import sklearn
import tensorflow as tf

SEED = 2026
CLASSES = string.digits + string.ascii_uppercase
NUM_CLASSES = len(CLASSES)
IMAGE_SIZE = 32
SAMPLES_PER_CLASS = {'train': 400, 'validation': 100, 'test': 100}
BATCH_SIZE = 64
MAX_EPOCHS = 30
RUN_TRAIN = True

random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)
try:
    tf.config.experimental.enable_op_determinism()
except Exception as exc:
    print('Déterminisme complet non garanti :', type(exc).__name__)

BASE_OUTPUT = Path('/content/platevision_v3_runs') if Path('/content').exists() else Path.cwd() / 'platevision_v3_runs'
BASE_OUTPUT.mkdir(parents=True, exist_ok=True)
run_id = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ') + '_' + uuid.uuid4().hex[:8]
RUN_DIR = BASE_OUTPUT / run_id
RUN_DIR.mkdir(parents=True, exist_ok=False)
CHECKPOINT = RUN_DIR / 'platevision_cnn_v3.keras'

print('Python :', sys.version.split()[0], '| TensorFlow :', tf.__version__, '| scikit-learn :', sklearn.__version__)
print('GPU :', tf.config.list_physical_devices('GPU'))
print('Classes :', CLASSES, '| Nombre :', NUM_CLASSES)
print('Dossier V3 :', RUN_DIR)
assert NUM_CLASSES == 36

## 2 — Inventaire des polices et séparation par groupes

Nous sélectionnons des familles orientées alphabet latin et groupons leurs variantes apparentées (ex. *Liberation Sans* et *Liberation Mono* dans le groupe *Liberation*). Les **groupes**, et non les images, sont attribués à train / validation / test. Les noms réellement disponibles sont imprimés avant la génération.

Le jeu de données reste dépendant des polices installées sur la machine. Un manifeste détaillé enregistre les fichiers sélectionnés afin que le protocole soit auditable.

In [ ]:
# Familles latines courantes sur Google Colab. Il ne s'agit pas d'une garantie de
# ressemblance avec les polices de véritables plaques d'immatriculation.
LATIN_FAMILY_PREFIXES = (
    'DejaVu Sans', 'Liberation Sans', 'Liberation Mono', 'Lato',
    'Roboto', 'Open Sans', 'Arimo', 'Carlito', 'Clear Sans',
    'Noto Sans', 'PT Sans', 'FreeSans', 'Cousine',
)

# Les variantes d'un même ensemble esthétique ne doivent pas franchir la frontière train/test.
def family_group(family: str) -> str:
    value = family.casefold()
    for prefix, key in (
        ('dejavu', 'DejaVu'), ('liberation', 'Liberation'),
        ('noto', 'Noto'), ('roboto', 'Roboto'),
        ('open sans', 'Open Sans'), ('pt sans', 'PT Sans'),
        ('clear sans', 'Clear Sans'),
    ):
        if value.startswith(prefix):
            return key
    return family

FONT_ROOT = Path('/usr/share/fonts/truetype')
if not FONT_ROOT.exists():
    raise RuntimeError('Polices Linux introuvables : exécuter ce notebook dans Google Colab.')

font_groups = defaultdict(list)
font_metadata = {}
for path in sorted(FONT_ROOT.rglob('*.ttf')):
    if any(part in path.name.lower() for part in ('italic', 'oblique', 'script', 'symbol')):
        continue
    # Préfiltrer les noms de fichiers pour éviter d'ouvrir des milliers de fontes
    # multi-scriptes sans rapport avec les caractères latins recherchés.
    stem = path.stem.casefold()
    if not any(stem.startswith(prefix) for prefix in (
        'dejavu', 'liberation', 'lato', 'roboto', 'opensans', 'arimo',
        'carlito', 'clearsans', 'notosans-', 'notosansdisplay',
        'notosansmono', 'pts', 'ptn', 'freesans', 'cousine',
    )):
        continue
    try:
        font = ImageFont.truetype(str(path), size=30)
        family, style = font.getname()
        allowed_families = {
            'DejaVu Sans', 'DejaVu Sans Condensed', 'DejaVu Sans Mono',
            'Liberation Sans', 'Liberation Sans Narrow', 'Liberation Mono',
            'Lato', 'Roboto', 'Roboto Condensed', 'Open Sans',
            'Open Sans Condensed', 'Arimo', 'Carlito', 'Clear Sans',
            'Clear Sans Light', 'Clear Sans Medium', 'Noto Sans',
            'Noto Sans Display', 'Noto Sans Mono', 'PT Sans',
            'PT Sans Narrow', 'FreeSans', 'Cousine',
        }
        if family not in allowed_families:
            continue
        # Rejeter les fichiers où les caractères demandés ne sont pas tous rendables.
        # La présence d'une vraie boîte d'encre est un contrôle minimal (non exhaustif).
        if not all(font.getbbox(char) is not None for char in CLASSES):
            continue
    except (OSError, ValueError):
        continue
    group = family_group(family)
    font_groups[group].append(path)
    font_metadata[str(path)] = {'family': family, 'style': style, 'group': group}

GROUPS = sorted(font_groups)
if len(GROUPS) < 3:
    raise RuntimeError(
        f'Seulement {len(GROUPS)} groupes de polices utilisables ({GROUPS}). '
        'Il en faut au moins 3 ; installez des polices latines supplémentaires avant de continuer.'
    )

rng = np.random.default_rng(SEED)
shuffled_groups = list(rng.permutation(GROUPS))
num_eval_groups = max(1, round(len(GROUPS) * 0.2))
if 2 * num_eval_groups >= len(GROUPS):
    num_eval_groups = 1
split_groups = {
    'validation': sorted(shuffled_groups[:num_eval_groups]),
    'test': sorted(shuffled_groups[num_eval_groups:2*num_eval_groups]),
    'train': sorted(shuffled_groups[2*num_eval_groups:]),
}
assert all(split_groups.values())
assert not (set(split_groups['train']) & set(split_groups['validation']))
assert not (set(split_groups['train']) & set(split_groups['test']))
assert not (set(split_groups['validation']) & set(split_groups['test']))
assert set().union(*(set(v) for v in split_groups.values())) == set(GROUPS)

for partition in ('train', 'validation', 'test'):
    selected = split_groups[partition]
    print(f'{partition:10}: groupes={selected} | fichiers={sum(len(font_groups[g]) for g in selected)}')
if len(split_groups['train']) < 3:
    warnings.warn('Peu de groupes à entraîner : la généralisation et les scores peuvent être instables.')
print('\nListe des familles et fichiers :')
for group in GROUPS:
    print(f'  {group}: {len(font_groups[group])} fichier(s)')

## 3 — Génération d'images et contrôles d'intégrité

Un caractère est dessiné dans une image 48×48 puis réduit à **32×32**, avec légères variations de position, rotation, intensité, bruit et flou. Le format final du CNN reste **`float32` dans `[0,1]`, forme `(32,32,1)`, caractère clair sur fond sombre**.

Les tirages de train, validation et test emploient **des groupes de polices disjoints** et des graines indépendantes. Nous contrôlons aussi les doublons **exacts** entre les partitions ; cela n'exclut pas tous les doublons visuellement proches.

In [ ]:
def make_character_image(char: str, font_path: Path, rng: np.random.Generator) -> np.ndarray:
    font = ImageFont.truetype(str(font_path), size=int(rng.integers(26, 38)))
    canvas = Image.new('L', (48, 48), color=255)
    draw = ImageDraw.Draw(canvas)
    left, top, right, bottom = draw.textbbox((0, 0), char, font=font)
    width, height = right - left, bottom - top
    dx, dy = rng.integers(-3, 4, size=2)
    origin = ((48 - width) / 2 - left + int(dx), (48 - height) / 2 - top + int(dy))
    gray_ink = int(rng.integers(0, 51))
    draw.text(origin, char, font=font, fill=gray_ink)
    canvas = canvas.rotate(float(rng.uniform(-9, 9)), resample=Image.Resampling.BICUBIC, fillcolor=255)
    canvas = canvas.resize((IMAGE_SIZE, IMAGE_SIZE), resample=Image.Resampling.BICUBIC)
    if rng.random() < 0.30:
        canvas = canvas.filter(ImageFilter.GaussianBlur(radius=float(rng.uniform(0.15, 0.65))))
    if rng.random() < 0.55:
        canvas = ImageEnhance.Contrast(canvas).enhance(float(rng.uniform(0.78, 1.22)))
    if rng.random() < 0.40:
        canvas = ImageEnhance.Brightness(canvas).enhance(float(rng.uniform(0.88, 1.12)))
    array = np.asarray(canvas, dtype=np.float32)
    if rng.random() < 0.50:
        array += rng.normal(0, rng.uniform(1.0, 7.0), size=array.shape)
    if rng.random() < 0.12:
        # JPEG modéré : simulation d'une compression sans utiliser de données réelles.
        temp = Image.fromarray(np.clip(array, 0, 255).astype(np.uint8), mode='L')
        buf = BytesIO()
        temp.save(buf, format='JPEG', quality=int(rng.integers(55, 91)))
        buf.seek(0)
        with Image.open(buf) as decoded:
            array = np.asarray(decoded.convert('L'), dtype=np.float32)
    return np.clip(array, 0, 255).astype(np.uint8)


def generate_dataset(partition: str, n_per_class: int, seed: int):
    local_rng = np.random.default_rng(seed)
    files = [p for group in split_groups[partition] for p in font_groups[group]]
    assert files, f'Pas de police pour {partition}'
    count = n_per_class * NUM_CLASSES
    # Mémoire : 21 600 × 32 × 32 uint8 pour la génération, puis float32 pour le CNN.
    raw = np.empty((count, IMAGE_SIZE, IMAGE_SIZE), dtype=np.uint8)
    labels = np.empty(count, dtype=np.int32)
    offset = 0
    for label, char in enumerate(CLASSES):
        for _ in range(n_per_class):
            font_path = files[int(local_rng.integers(0, len(files)))]
            raw[offset] = make_character_image(char, font_path, local_rng)
            labels[offset] = label
            offset += 1
    order = local_rng.permutation(count)
    raw, labels = raw[order], labels[order]
    images = (1.0 - raw.astype(np.float32) / 255.0)[..., np.newaxis]
    assert images.shape == (count, IMAGE_SIZE, IMAGE_SIZE, 1)
    assert np.all(np.isfinite(images)) and 0.0 <= float(images.min()) <= float(images.max()) <= 1.0
    assert np.array_equal(np.bincount(labels, minlength=NUM_CLASSES), np.full(NUM_CLASSES, n_per_class))
    print(f'{partition:10}: {count:,} exemples, {n_per_class} / classe, {images.nbytes/2**20:.1f} Mio float32')
    return images, labels, raw

X_train, y_train, raw_train = generate_dataset('train', SAMPLES_PER_CLASS['train'], SEED + 101)
X_val, y_val, raw_val = generate_dataset('validation', SAMPLES_PER_CLASS['validation'], SEED + 202)
X_test, y_test, raw_test = generate_dataset('test', SAMPLES_PER_CLASS['test'], SEED + 303)

# Vérification sur les pixels uint8 réellement produits, avant la normalisation.
def fingerprint_set(images):
    return {hashlib.sha256(image.tobytes()).digest() for image in images}

hashes = {
    'train': fingerprint_set(raw_train),
    'validation': fingerprint_set(raw_val),
    'test': fingerprint_set(raw_test),
}
for a, b in [('train', 'validation'), ('train', 'test'), ('validation', 'test')]:
    overlap = len(hashes[a] & hashes[b])
    print(f'Doublons exacts {a} / {b}: {overlap}')
    if overlap:
        raise RuntimeError(f'{overlap} doublon(s) entre {a} et {b}. Interrompre et revoir le générateur.')

manifest = {
    'experiment': 'PlateVision CNN V3 - caractères synthétiques isolés',
    'seed': SEED,
    'classes': CLASSES,
    'image_shape': [32, 32, 1],
    'preprocessing': 'PIL grayscale uint8 -> 1.0 - image / 255.0 -> float32 [0,1]; caractères clairs sur fond sombre',
    'samples_per_class': SAMPLES_PER_CLASS,
    'split_groups': split_groups,
    'font_metadata': font_metadata,
    'font_files_used': {split: [str(p) for g in groups for p in font_groups[g]] for split, groups in split_groups.items()},
    'numpy': np.__version__, 'tensorflow': tf.__version__, 'sklearn': sklearn.__version__,
    'test_evaluation_default': False,
    'disclaimer': 'Synthétique uniquement ; split par groupes de polices ; comparaisons V2 non directes.',
}
(RUN_DIR / 'manifest_v3.json').write_text(json.dumps(manifest, indent=2, ensure_ascii=False), encoding='utf-8')
print('Manifeste enregistré :', RUN_DIR / 'manifest_v3.json')

## 4 — Vérification visuelle des données **d'entraînement**

Observer ici si les caractères ressemblent réellement aux entrées attendues. Si des caractères sont absents, tronqués ou inversés, **corriger avant l'entraînement**. Ne pas afficher les exemples du jeu de test pendant la conception.

In [ ]:
fig, axes = plt.subplots(4, 9, figsize=(13, 6))
for i, ax in enumerate(axes.flat):
    label = i  # un exemple de chacune des 36 classes
    index = int(np.flatnonzero(y_train == label)[0])
    ax.imshow(X_train[index, :, :, 0], cmap='gray', vmin=0, vmax=1)
    ax.set_title(CLASSES[label], fontsize=10)
    ax.axis('off')
fig.suptitle('V3 — 36 classes (aperçu TRAIN, caractère clair sur fond sombre)')
plt.tight_layout()
plt.savefig(RUN_DIR / 'training_samples_v3.png', dpi=140, bbox_inches='tight')
plt.show()

## 5 — Architecture CNN V3

Le réseau est volontairement plus compact que V2 : convolution + normalisation + pooling, puis une tête de classification de 36 classes. Les augmentations intégrées au modèle ne s'appliquent que pendant l'entraînement.

**Important :** les entrées sont déjà normalisées dans `[0,1]` ; il n'y a donc **aucun** `Rescaling(1/255)` additionnel ni `RandomContrast` configuré pour `[0,255]`.

In [ ]:
from tensorflow.keras import layers, models

def build_cnn_v3():
    augmentation = models.Sequential([
        layers.RandomRotation(0.025, fill_mode='constant', fill_value=0.0),
        layers.RandomTranslation(0.04, 0.04, fill_mode='constant', fill_value=0.0),
        layers.RandomZoom(0.04, fill_mode='constant', fill_value=0.0),
    ], name='augmentation_train_only')
    inputs = layers.Input(shape=(IMAGE_SIZE, IMAGE_SIZE, 1), name='character_32x32')
    x = augmentation(inputs)
    for filters in (32, 64, 128):
        x = layers.Conv2D(filters, 3, padding='same', use_bias=False)(x)
        x = layers.BatchNormalization()(x)
        x = layers.Activation('relu')(x)
        x = layers.MaxPooling2D()(x)
    x = layers.Conv2D(128, 3, padding='same', use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.Flatten()(x)
    x = layers.Dense(128, activation='relu')(x)
    x = layers.Dropout(0.4)(x)
    outputs = layers.Dense(NUM_CLASSES, activation='softmax', name='class_probabilities')(x)
    result = models.Model(inputs, outputs, name='platevision_cnn_v3')
    result.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=5e-4),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy'],
    )
    return result

model_v3 = build_cnn_v3()
model_v3.summary()
print('Paramètres V3 :', model_v3.count_params())
assert model_v3.input_shape == (None, 32, 32, 1)
assert model_v3.output_shape == (None, 36)

## 6 — Entraînement V3 (activé)

Sur GPU, l'entraînement utilise un arrêt anticipé fondé sur `val_loss` et sauvegarde le **meilleur checkpoint de validation**. Aucun fichier V1/V2 n'est utilisé. Le dossier unique de la session contient le modèle, l'historique et les métadonnées.

L'exécution CPU reste possible, mais peut être nettement plus lente. L'accuracy affichée pendant `fit()` n'est **pas** une mesure sur des plaques réelles.

In [ ]:
HISTORY_FILE = RUN_DIR / 'history_v3.json'
if RUN_TRAIN:
    callbacks = [
        tf.keras.callbacks.ModelCheckpoint(str(CHECKPOINT), monitor='val_loss', save_best_only=True, mode='min', verbose=1),
        tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=6, mode='min', restore_best_weights=True),
        tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-5, verbose=1),
        tf.keras.callbacks.CSVLogger(str(RUN_DIR / 'epochs_v3.csv')),
    ]
    training = model_v3.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        batch_size=BATCH_SIZE,
        epochs=MAX_EPOCHS,
        callbacks=callbacks,
        verbose=1,
        shuffle=True,
    )
    history = {k: [float(v) for v in values] for k, values in training.history.items()}
    HISTORY_FILE.write_text(json.dumps(history, indent=2), encoding='utf-8')
    print('Entraînement terminé. Meilleur checkpoint :', CHECKPOINT)
else:
    print('RUN_TRAIN=False : entraînement désactivé. Aucun checkpoint généré.')

## 7 — Analyse de la validation (pour décider si l'expérience est satisfaisante)

**Seule la validation** sert ici aux choix d'architecture et d'hyperparamètres. Les confusions difficiles, par exemple `J`, `3`, `1` et `S`, sont inspectées sur ce jeu. Le **test reste intact**.

In [ ]:
if not CHECKPOINT.exists():
    print('Pas de checkpoint V3 : aucune évaluation possible.')
else:
    best_v3 = tf.keras.models.load_model(str(CHECKPOINT))
    val_loss, val_accuracy = best_v3.evaluate(X_val, y_val, verbose=0)
    val_predictions = np.argmax(best_v3.predict(X_val, batch_size=256, verbose=0), axis=1)
    val_report = classification_report(
        y_val, val_predictions,
        labels=np.arange(NUM_CLASSES), target_names=list(CLASSES),
        output_dict=True, zero_division=0,
    )
    val_metrics = {
        'validation_loss': float(val_loss), 'validation_accuracy': float(val_accuracy),
        'per_class_report': val_report,
        'confusion_matrix': confusion_matrix(y_val, val_predictions, labels=np.arange(NUM_CLASSES)).tolist(),
    }
    (RUN_DIR / 'validation_metrics_v3.json').write_text(json.dumps(val_metrics, indent=2), encoding='utf-8')
    print(f'Validation V3 : accuracy={val_accuracy:.2%} | loss={val_loss:.4f}')
    print(classification_report(y_val, val_predictions, labels=np.arange(NUM_CLASSES), target_names=list(CLASSES), zero_division=0))
    if HISTORY_FILE.exists():
        hist = json.loads(HISTORY_FILE.read_text(encoding='utf-8'))
        fig, ax = plt.subplots(1, 2, figsize=(12, 4))
        for name, axis in [('loss', ax[0]), ('accuracy', ax[1])]:
            axis.plot(hist[name], label='Entraînement')
            axis.plot(hist['val_' + name], label='Validation')
            axis.set(title=name.capitalize(), xlabel='Époque')
            axis.legend()
        plt.tight_layout()
        plt.savefig(RUN_DIR / 'training_curves_v3.png', dpi=150, bbox_inches='tight')
        plt.show()

## 8 — Test final **verrouillé par défaut**

Exécuter **une seule fois**, après avoir figé architecture, paramètres et augmentations sur validation. Mettre `RUN_FINAL_TEST = True` **au début de la cellule de test ci-dessous**, puis réexécuter **uniquement cette cellule** (ne pas relancer toute la formation ni la cellule de configuration, qui crée un nouveau dossier de session). Un verrou local empêche sa répétition accidentelle dans le même dossier.

⚠️ **Ne pas réutiliser ce test pour régler les hyperparamètres.** Un jeu de test indépendant de photos réelles sera nécessaire pour caractériser la performance en conditions réelles.

In [ ]:
RUN_FINAL_TEST = False  # Modifier ICI à True uniquement après avoir figé V3 sur validation.

test_file = RUN_DIR / 'test_metrics_v3.json'
if not RUN_FINAL_TEST:
    print('Test final réservé : RUN_FINAL_TEST=False. Aucune métrique test calculée.')
elif test_file.exists():
    print('Test déjà réalisé pour cette session. Résultats :', test_file)
elif not CHECKPOINT.exists():
    print('Impossible : aucun checkpoint V3 entraîné.')
else:
    model_final = tf.keras.models.load_model(str(CHECKPOINT))
    test_loss, test_accuracy = model_final.evaluate(X_test, y_test, verbose=0)
    test_pred = np.argmax(model_final.predict(X_test, batch_size=256, verbose=0), axis=1)
    test_report = classification_report(
        y_test, test_pred, labels=np.arange(NUM_CLASSES), target_names=list(CLASSES),
        output_dict=True, zero_division=0,
    )
    test_metrics = {
        'test_loss': float(test_loss), 'test_accuracy': float(test_accuracy),
        'per_class_report': test_report,
        'confusion_matrix': confusion_matrix(y_test, test_pred, labels=np.arange(NUM_CLASSES)).tolist(),
        'warning': 'Jeu de test synthétique ; pas une mesure de précision sur les plaques réelles.'
    }
    test_file.write_text(json.dumps(test_metrics, indent=2), encoding='utf-8')
    print(f'TEST FINAL V3 : accuracy={test_accuracy:.2%} | loss={test_loss:.4f}')
    print(classification_report(y_test, test_pred, labels=np.arange(NUM_CLASSES), target_names=list(CLASSES), zero_division=0))

## 9 — Export pour Codex / application Streamlit

Télécharger l'archive de la session : elle comprend le modèle `.keras`, `classes.json`, les courbes, les métriques de validation, le manifeste et, **si exécutées**, les métriques du test final. Le modèle attend des **caractères déjà segmentés**, prétraités en 32×32 niveaux de gris puis inversés et normalisés.

Les checkpoints sont enregistrés sous `/content` et peuvent disparaître à la fermeture de Colab : **télécharge l'archive avant de fermer la session**. Les images synthétiques complètes ne sont pas incluses dans l'export (elles sont régénérables avec les graines et les polices répertoriées).

In [ ]:
classes_file = RUN_DIR / 'classes.json'
classes_file.write_text(json.dumps(list(CLASSES), indent=2), encoding='utf-8')
preprocessing_file = RUN_DIR / 'INFERENCE_CONTRACT.txt'
preprocessing_file.write_text(
    'PlateVision CNN V3 — classification de CARACTÈRES isolés.\n'
    'Entrée: (batch, 32, 32, 1), float32, [0,1].\n'
    'Préparation: image du caractère en GRAYSCALE, redimensionner 32x32, '
    'puis 1.0 - grayscale/255.0 (caractère clair sur fond sombre).\n'
    'Classes par indice: 0123456789ABCDEFGHIJKLMNOPQRSTUVWXYZ.\n'
    'Les augmentations sont désactivées lors de predict().\n'
    'Ne pas interpréter la softmax comme une confiance calibrée.\n',
    encoding='utf-8',
)

archive_path = RUN_DIR.parent / (RUN_DIR.name + '_export.zip')
if CHECKPOINT.exists():
    with zipfile.ZipFile(archive_path, 'w', compression=zipfile.ZIP_DEFLATED, compresslevel=5) as archive:
        for path in sorted(RUN_DIR.iterdir()):
            if path.is_file():
                archive.write(path, arcname=path.name)
    print('Archive prête :', archive_path, '|', round(archive_path.stat().st_size / (2**20), 2), 'Mio')
    try:
        from google.colab import files
        files.download(str(archive_path))
    except ImportError:
        print('Hors Colab : récupérer le fichier dans le dossier indiqué.')
else:
    print('Le modèle n’est pas entraîné : export reporté.')

## 10 — Ce que signifie (et ne signifie pas) le résultat

- **Oui :** comparaison des architectures sur des caractères synthétiques, avec un split sans recouvrement de groupes de polices et des métriques de validation documentées.
- **Non :** preuve que V3 dépasse V2 sur le même benchmark. Le générateur, les groupes de polices et les partitions ont changé ; comparer directement `80,65 %` (V2 ancien test) à une future accuracy V3 serait trompeur.
- **Non :** accuracy end-to-end sur des plaques photographiées. La détection, la segmentation et l'OCR restent des étapes distinctes.

**Suite recommandée :** transmettre l'archive exportée à Codex, mesurer V2 et V3 sur un même jeu de caractères réels correctement annotés et indépendant de la conception des deux systèmes, puis intégrer la V3 seulement si elle apporte un gain validé. La photographie connue `J389NLT` ne doit pas servir à choisir les paramètres de V3.